In [1]:
# Pre-training SoRL on arithmatic generalization dataset
# ------------------------------------------------------ 
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

assert BOS_TOKEN_ID == 20, "BOS_TOKEN_ID must be set to 20 (go to 'sorl/gat_sim.py' to change it)"
gat_config = GATConfig(
    vocab_sizes=[20, 6],  # 6 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

Generate multiplication data: 

```python data/arithmetic.py```

In [2]:
# ---- Arithmatic generalization dataset loader ----
from sorl.arithmetic import data_generator
from sorl.arithmetic import DigitTokenizer

# --- tokenizer ---
tokenizer = DigitTokenizer()

# --- data loader ---
train_loader = data_generator(filename_pattern="data/multiplication/multiplication_train.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/multiplication/multiplication_val_id.bin", sequence_length=64, device="cpu")

In [4]:
from sorl.neo_utils import sorl_search, sorl_search_v2, compute_loss, sorl_evaluate, compute_vocab_utilization_rate
from sorl.gapt import GatedPhaseTransition

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

# memory_span = 1
memory_span = 25 # control amount of memory stored in trajectory token (rest is stored in abstraction token)
attn_blocksize = 1792
K = 4  # abstraction ratio
n = 2  # number of rollout
temperatures = torch.tensor([0.5, 10.0], device=model.device)
max_iterations = 2  # of denoising steps for generating abstraction token (in parallel)
alpha_loss = 0.1 # weight on abstraction perplexity (training)
alpha_sel = 0.2  # weight on curiosity reward (during selection)
sel_mode = "vocab_util" # selection mode (abs_ppt / vocab_util)
n_eval = 4  # number of rollout for evaluation
temperatures_eval = torch.tensor([0.5, 5.0, 5.0, 5.0], device=model.device) # temperature for evaluation

num_steps = 500
gapt = GatedPhaseTransition(p_m=10)

for step in range(num_steps): 
 
    optimizer.zero_grad()

    tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search_v2(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures,
                                                               truncate_seq_len=False, alpha_select=alpha_sel, select_mode=sel_mode)
    
    # --- compute loss ---
    traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize)

                                       
    loss = gapt.step(traj_loss, alpha_loss * abs_loss, verbose=True) # gated phase transition loss adaptor
    # loss = traj_loss + alpha_loss * abs_loss # simple weighed sum 

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(search_tokens, model, n=n_eval, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                   truncate_seq_len=False)
            traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize)
            vocab_util = compute_vocab_utilization_rate(val_tokens, model)
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}% | vocab util: {vocab_util * 100:.2f}%")

validation step 0 | traj_loss: 3.21 | abs_loss: 3.27 | search adv: 0.04% | vocab util: 83.33%
validation step 2 | traj_loss: 3.10 | abs_loss: 3.37 | search adv: 0.04% | vocab util: 83.33%
validation step 4 | traj_loss: 2.95 | abs_loss: 3.62 | search adv: -0.02% | vocab util: 83.33%
validation step 6 | traj_loss: 2.82 | abs_loss: 3.86 | search adv: 0.00% | vocab util: 83.33%
validation step 8 | traj_loss: 2.74 | abs_loss: 4.08 | search adv: -0.06% | vocab util: 83.33%
validation step 10 | traj_loss: 2.62 | abs_loss: 4.34 | search adv: 0.06% | vocab util: 83.33%
validation step 12 | traj_loss: 2.57 | abs_loss: 4.30 | search adv: -0.02% | vocab util: 83.33%
validation step 14 | traj_loss: 2.43 | abs_loss: 4.52 | search adv: 0.00% | vocab util: 83.33%
validation step 16 | traj_loss: 2.40 | abs_loss: 4.64 | search adv: -0.01% | vocab util: 83.33%
validation step 18 | traj_loss: 2.28 | abs_loss: 4.88 | search adv: 0.12% | vocab util: 83.33%
validation step 20 | traj_loss: 2.24 | abs_loss: 5.

In [5]:
# generate function implementation 
# ----------------------------------
from sorl.neo_utils import generate
from sorl.arithmetic import process_query, check_answer

K = 5
tokens = next(val_loader)
idx, answer_idx = process_query(tokens)

print(f"init   | idx: {idx[0].tolist()} | question: {tokenizer.decode(idx[0].tolist()[1:-1])}")
for i in range(len(answer_idx[0])*2): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, 
                   temperature=temperatures_eval[0])
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    print(f"                         idx : {idx[0].tolist()}")

is_correct, pred_answer, true_answer = check_answer(idx_without_abstraction, answer_idx, tokenizer)
print(f"is_correct: {is_correct} | pred_answer: {pred_answer} | true_answer: {true_answer}")

init   | idx: [0, 15, 4, 2, 4, 18, 4, 3] | question: 5 x 8 
step 1 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4]
                         idx : [0, 15, 4, 2, 4, 21, 18, 4, 3, 4]
step 2 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 15]
                         idx : [0, 15, 4, 2, 4, 24, 18, 4, 3, 4, 15]
step 3 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 15, 12]
                         idx : [0, 15, 4, 2, 4, 21, 18, 4, 3, 4, 15, 23, 12]
step 4 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 15, 12, 1]
                         idx : [0, 15, 4, 2, 4, 21, 18, 4, 3, 4, 15, 24, 12, 1]
step 5 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 15, 12, 1, 0]
                         idx : [0, 15, 4, 2, 4, 22, 18, 4, 3, 4, 15, 23, 12, 1, 0]
step 6 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 15, 12, 1, 0, 13]
                         idx : [0, 15, 4, 2, 4, 21, 18, 4, 3, 4, 15, 24, 12, 1, 0, 13]
step 7 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 